# RealData Graph Retrieval and NL-to-Cypher for PostgreSQL AGE

This notebook is the retrieval/query pipeline. It reuses the safe Apache AGE schema extraction and read-only NL-to-Cypher pattern from `nl2cypher_forpsql.ipynb`, pointed at `realdata_knowledge_spine`.


In [ ]:
# Install only if your notebook environment does not already have these packages.
# ! pip install openai "psycopg[binary]" python-dotenv


In [ ]:
import csv
import json
import os
import re

import psycopg
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(override=True)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

GRAPH_NAME = "realdata_knowledge_spine"


client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://ciaiciath2-foundry-dev.cognitiveservices.azure.com/",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

models = ["gpt-5.6-luna", "gpt-5.4-mini"]
NL2CYPHER_MODEL = models[0]
ANSWER_MODEL = models[0]

In [2]:
def connect_postgres():
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


def open_age_connection():
    conn = connect_postgres()
    try:
        with conn.cursor() as cursor:
            init_age(cursor)
        conn.commit()
        return conn
    except Exception:
        conn.rollback()
        conn.close()
        raise


# Smoke test only. Query cells open their own short-lived connections.
test_conn = open_age_connection()
test_conn.close()
print("Connected to PostgreSQL AGE graph:", GRAPH_NAME)


Connected to PostgreSQL AGE graph: payer_policy_knowledge_graph


In [3]:
def normalize_agtype(value):
    if value is None:
        return None
    if isinstance(value, (list, dict, int, float, bool)):
        return value
    text = str(value).strip()
    try:
        return json.loads(text)
    except Exception:
        return text.strip('"')


In [4]:
def get_age_node_schema(conn, graph_name):
    nodes = {}
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (n)
            RETURN DISTINCT labels(n), keys(n)
        $$) AS (labels agtype, properties agtype);
        """)
        rows = cursor.fetchall()

    for labels_value, properties_value in rows:
        labels = normalize_agtype(labels_value)
        properties = normalize_agtype(properties_value)
        if not isinstance(labels, list):
            labels = [labels]
        if not isinstance(properties, list):
            properties = [properties]
        for label in labels:
            nodes.setdefault(str(label), set()).update(str(prop) for prop in properties)
    return {label: sorted(properties) for label, properties in nodes.items()}


def get_age_relationship_schema(conn, graph_name):
    relationships = []
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (a)-[r]->(b)
            RETURN DISTINCT labels(a), type(r), labels(b)
        $$) AS (source_labels agtype, relationship agtype, target_labels agtype);
        """)
        rows = cursor.fetchall()

    for source_value, relationship_value, target_value in rows:
        relationships.append({
            "source": normalize_agtype(source_value),
            "relationship": normalize_agtype(relationship_value),
            "target": normalize_agtype(target_value),
        })
    return relationships


def get_age_graph_schema(conn, graph_name):
    return {
        "nodes": get_age_node_schema(conn, graph_name),
        "relationships": get_age_relationship_schema(conn, graph_name),
    }


In [5]:
def build_schema_text(schema):
    lines = ["NODE LABELS AND PROPERTIES"]
    for label, properties in schema["nodes"].items():
        lines.append(f"\nNode: {label}")
        lines.append("Properties: " + ", ".join(properties))

    lines.append("\nRELATIONSHIPS")
    for relation in schema["relationships"]:
        source = relation["source"]
        target = relation["target"]
        if isinstance(source, list):
            source = ", ".join(source)
        if isinstance(target, list):
            target = ", ".join(target)
        lines.append(f"({source})-[:{relation['relationship']}]->({target})")
    return "\n".join(lines)


schema_conn = open_age_connection()
try:
    age_schema = get_age_graph_schema(schema_conn, GRAPH_NAME)
finally:
    schema_conn.close()

GRAPH_SCHEMA = build_schema_text(age_schema)
print(GRAPH_SCHEMA)


NODE LABELS AND PROPERTIES

Node: Artifact
Properties: artifact_id, artifact_type, content_version, last_reviewed, layer, node_id, review_status, schema_version, scope, search_text, source_file, subject_area, title, use_case_id

Node: Concept
Properties: adjacent_concepts, ai_routing_note, artifact_id, concept_id, concept_version, definition, disambiguation_triggers, flags, knowledge_latest_as_of, knowledge_source, monitoring_needed, name, node_id, retrieval_tags, review_date, search_text, source_file, synonyms, vendor_naming, what_it_is_not

Node: Step
Properties: also_known_as, artifact_id, concept_refs, disambiguation_triggers, execution_eligibility, execution_language, fallback, guardrails, input_contract, name, next_step, node_id, of_steps, output_contract, previous_step, produces, purpose, readiness, retrieval_tags, schema_bindings, search_text, source_file, stage_contract_version, step_id, step_number, uses_slots, variation_composition, variations, why_here

Node: Slot
Propertie

In [5]:
FORBIDDEN_CYPHER = [
    "CREATE",
    "MERGE",
    "DELETE",
    "DETACH",
    "SET",
    "REMOVE",
    "DROP",
    "LOAD CSV",
    "FOREACH",
    "CALL",
]

# Fallback search uses one normalized string property created during transformation.
SAFE_TEXT_SEARCH_PROPERTIES = ["search_text"]

UNSAFE_TEXT_FUNCTION_PROPERTIES = {
    "name",          # number in some YAML records
    "purpose",       # dict/list/string depending on file
    "domain",        # dict/string depending on file
    "use_case_id",   # list/string/null depending on file
    "synonyms",
    "retrieval_tags",
    "business_rules",
    "sql_rules",
    "narrative_rules",
    "routing",
    "records",
    "instructions",
    "fields",
    "relationships",
    "dq_standards",
}


def strip_string_literals(cypher):
    result = []
    in_quote = None
    escaped = False
    for char in cypher:
        if in_quote:
            if escaped:
                escaped = False
                continue
            if char == "\\":
                escaped = True
                continue
            if char == in_quote:
                in_quote = None
                result.append("''")
            continue
        if char in {"'", '"'}:
            in_quote = char
            continue
        result.append(char)
    return "".join(result)


def validate_read_only_cypher(cypher):
    normalized = strip_string_literals(cypher).upper().strip()
    if ";" in normalized:
        raise ValueError("Cypher must not include semicolons")
    for keyword in FORBIDDEN_CYPHER:
        if re.search(rf"\b{re.escape(keyword)}\b", normalized):
            raise ValueError(f"Unsafe Cypher detected: {keyword}")
    if not re.match(r"^(MATCH|WITH|UNWIND)\b", normalized):
        raise ValueError("Cypher must start with MATCH, WITH, or UNWIND")
    return True


def validate_age_compatible_cypher(cypher):
    literal_free = strip_string_literals(cypher)
    normalized = literal_free.upper().strip()

    # Apache AGE often fails on generated OPTIONAL MATCH chains. Keep this direct notebook strict.
    if re.search(r"\bOPTIONAL\s+MATCH\b", normalized):
        raise ValueError("OPTIONAL MATCH is disabled in this notebook; use plain MATCH or fallback search")

    # AGE toString()/toLower() are scalar-only. Generated queries commonly apply them to lists/maps.
    if re.search(r"\bTOSTRING\s*\(", normalized):
        raise ValueError("toString() is disabled because AGE only supports scalar arguments")

    for match in re.finditer(
        r"toLower\s*\(\s*(?:coalesce\s*\(\s*)?[A-Za-z_][A-Za-z0-9_]*\.([A-Za-z_][A-Za-z0-9_]*)",
        literal_free,
        flags=re.IGNORECASE,
    ):
        prop = match.group(1)
        if prop not in SAFE_TEXT_SEARCH_PROPERTIES:
            raise ValueError(f"toLower() used on unsafe or non-scalar property: {prop}")

    for prop in UNSAFE_TEXT_FUNCTION_PROPERTIES:
        if re.search(rf"\b(?:toLower|toString)\s*\([^)]*\.{re.escape(prop)}\b", literal_free, flags=re.IGNORECASE):
            raise ValueError(f"Text function used on unsafe or non-scalar property: {prop}")

    return True


def validate_column_name(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid column name: {name}")
    return name


In [6]:
def llm_json(messages, model):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


def validate_query_spec(query_spec):
    cypher = query_spec["cypher"].strip()
    columns = [validate_column_name(column) for column in query_spec["columns"]]
    validate_read_only_cypher(cypher)
    validate_age_compatible_cypher(cypher)
    return {"cypher": cypher, "columns": columns}


def generate_age_query(question, retry_context=None):
    retry_context = retry_context or ""
    messages = [
        {
            "role": "system",
            "content": """
You generate read-only Apache AGE Cypher queries.
Return JSON only with this exact shape:
{"cypher": "MATCH ... RETURN ...", "columns": ["column_1"]}

Rules:
- Use only the supplied schema.
- Never invent labels, properties, or relationships.
- Do not include the PostgreSQL SELECT FROM cypher wrapper.
- Do not use OPTIONAL MATCH in this notebook.
- Do not use toString().
- Every RETURN expression must have an explicit alias.
- Prefer scalar properties instead of full vertices or edges.
- For keyword search, prefer n.search_text CONTAINS '<lowercase term>'.
- search_text is a normalized lowercase text field created during graph transformation.
- Never call toLower() or toString() on name, purpose, domain, use_case_id,
  synonyms, retrieval_tags, business_rules, sql_rules, narrative_rules,
  routing, records, instructions, fields, relationships, or dq_standards.
- Use broad matching for business questions. Example: for MMM, search both
  mmm and market mix and marketing mix.
- Always add LIMIT 10 or less.
- Read-only Cypher only.
- Avoid using very short acronyms with
  search_text CONTAINS by themselves.

- CONTAINS performs substring matching.
  A short acronym such as "LOT" can match
  unrelated text such as "slot".

- When the question provides both an acronym
  and its expanded form, prioritize the
  expanded phrase.

- Example:
  LOT (Line of Therapy)

  Prefer:
  search_text CONTAINS 'line of therapy'

  rather than relying primarily on:
  search_text CONTAINS 'lot'

- Prefer exact semantic phrases over short
  substring terms whenever possible.
""".strip(),
        },
        {
            "role": "user",
            "content": f"GRAPH SCHEMA:\n{GRAPH_SCHEMA}\n\nQUESTION:\n{question}\n\nRETRY CONTEXT:\n{retry_context}",
        },
    ]
    result = llm_json(messages, NL2CYPHER_MODEL)
    return validate_query_spec(result)


In [7]:
def execute_age_query(query_spec):
    cypher = query_spec["cypher"]
    columns = [validate_column_name(column) for column in query_spec["columns"]]
    validate_read_only_cypher(cypher)
    validate_age_compatible_cypher(cypher)

    column_definition = ", ".join(f"{column} agtype" for column in columns)
    sql = f"""
    SELECT *
    FROM cypher('{GRAPH_NAME}', $$
        {cypher}
    $$) AS ({column_definition});
    """

    conn = open_age_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            rows = cursor.fetchall()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return [
        {column: normalize_agtype(value) for column, value in zip(columns, row)}
        for row in rows
    ]


In [8]:
# Best-practice fallback search:
# 1) Ask the model to extract concise search terms dynamically.
# 2) If that fails, use simple quoted-phrase/token extraction.
# 3) Search only n.search_text, which is built during transformation.
# No business synonym dictionary and no manual stop-word list are maintained in retrieval code.


def escape_cypher_string(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def normalize_search_terms(raw_terms, max_terms=12):
    terms = []
    seen = set()
    for term in raw_terms or []:
        normalized = re.sub(r"\s+", " ", str(term).lower()).strip()
        normalized = normalized.strip("'\"`.,;:!?()[]{}")
        if len(normalized) < 2:
            continue
        if normalized not in seen:
            terms.append(normalized)
            seen.add(normalized)
        if len(terms) >= max_terms:
            break
    return terms


def deterministic_search_terms(question, max_terms=12):
    text = question.lower()
    quoted_phrases = re.findall(r"['\"]([^'\"]{2,80})['\"]", text)
    tokens = re.findall(r"[a-zA-Z0-9][a-zA-Z0-9_-]*", text)

    phrases = []
    useful_tokens = [token for token in tokens if len(token) >= 3 or any(char.isdigit() for char in token)]
    for size in (3, 2):
        for index in range(0, max(0, len(useful_tokens) - size + 1)):
            phrase = " ".join(useful_tokens[index:index + size])
            if len(phrase) <= 80:
                phrases.append(phrase)

    return normalize_search_terms(quoted_phrases + phrases + useful_tokens, max_terms=max_terms)


def llm_search_terms(question, max_terms=12):
    messages = [
        {
            "role": "system",
            "content": """
Extract concise graph search terms from the user question.
Return JSON only: {"terms": ["term1", "term2"]}
Rules:
- Include important acronyms, entity names, methodology names, metric names, and likely synonyms.
- Include expansions for acronyms only when useful.
- Do not include generic filler words unless they are part of a named business term.
- Keep each term short: 1 to 5 words.
- Return at most 12 terms.
""".strip(),
        },
        {
            "role": "user",
            "content": f"QUESTION:\n{question}\n\nGRAPH SCHEMA SUMMARY:\n{GRAPH_SCHEMA[:4000]}",
        },
    ]
    result = llm_json(messages, NL2CYPHER_MODEL)
    return normalize_search_terms(result.get("terms", []), max_terms=max_terms)


def search_terms_from_question(question, max_terms=12):
    llm_terms = []
    try:
        llm_terms = llm_search_terms(question, max_terms=max_terms)
    except Exception as exc:
        print("Search-term LLM failed; using deterministic terms only:", exc)

    deterministic_terms = deterministic_search_terms(question, max_terms=max_terms)
    return normalize_search_terms(llm_terms + deterministic_terms, max_terms=max_terms)


def build_fallback_query(question, limit=8):
    terms = search_terms_from_question(question)
    if not terms:
        terms = [question.lower()[:80]]

    predicates = []
    for term in terms:
        literal = escape_cypher_string(term)
        predicates.append(f"n.search_text CONTAINS '{literal}'")

    where_clause = "\n        OR ".join(predicates) or "false"
    cypher = f"""
MATCH (n)
WHERE {where_clause}
RETURN
    labels(n) AS labels,
    n.node_id AS node_id,
    n.title AS title,
    n.definition AS definition,
    n.description AS description,
    n.concept_id AS concept_id,
    n.entity_id AS entity_id,
    n.artifact_id AS artifact_id,
    n.artifact_type AS artifact_type,
    n.source_file AS source_file
LIMIT {limit}
""".strip()

    query_spec = validate_query_spec({
        "cypher": cypher,
        "columns": [
            "labels", "node_id", "title", "definition", "description",
            "concept_id", "entity_id", "artifact_id", "artifact_type", "source_file",
        ],
    })
    query_spec["search_terms"] = terms
    return query_spec


def compact_value(value, max_chars=900):
    if value is None:
        return None
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False, default=str)
    else:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + "... [truncated]"
    return text


def flatten_value(value):

    if value is None:
        return ""

    if isinstance(value, dict):
        return " ".join(
            flatten_value(v)
            for v in value.values()
        )

    if isinstance(value, list):
        return " ".join(
            flatten_value(v)
            for v in value
        )

    return str(value)


def extract_cypher_search_terms(cypher):

    terms = re.findall(
        r"CONTAINS\s+'([^']+)'",
        cypher,
        flags=re.IGNORECASE,
    )

    return [
        term.lower().strip()
        for term in terms
        if term.strip()
    ]


def row_relevance_score(
    row,
    search_terms
):

    full_text = flatten_value(
        row
    ).lower()

    important_text = " ".join(
        flatten_value(
            row.get(key)
        )
        for key in [
            "concept_name",
            "name",
            "title",
            "synonyms",
        ]
    ).lower()

    score = 0

    for term in search_terms:

        term = term.lower()

        # Short acronym such as LOT.
        # Require a complete word match.
        if (
            len(term) <= 4
            and " " not in term
        ):

            pattern = (
                rf"(?<![a-z0-9])"
                rf"{re.escape(term)}"
                rf"(?![a-z0-9])"
            )

            if re.search(
                pattern,
                important_text,
            ):
                score += 10

            elif re.search(
                pattern,
                full_text,
            ):
                score += 3

        # Longer phrase such as
        # "line of therapy"
        else:

            if term in important_text:
                score += 12

            elif term in full_text:
                score += 5

    return score

def compact_query_result(
    rows,
    cypher=None,
    max_rows=10,
    max_chars=1500,
):

    selected_rows = rows

    if cypher:

        search_terms = (
            extract_cypher_search_terms(
                cypher
            )
        )

        if search_terms:

            selected_rows = sorted(
                rows,
                key=lambda row:
                    row_relevance_score(
                        row,
                        search_terms,
                    ),
                reverse=True,
            )

    return [
        {
            key: compact_value(
                value,
                max_chars=max_chars,
            )
            for key, value
            in row.items()
        }
        for row
        in selected_rows[:max_rows]
    ]


def generate_answer(
    question,
    query_result,
    cypher=None,
):

    # Deterministic empty-result handling
    if not query_result:
        return "No matching data was found."

    compact_result = (
        compact_query_result(
            query_result,
            cypher=cypher,
        )
    )

    messages = [
        {
            "role": "system",
            "content": """
Answer using only the supplied graph query result.

The database returned matching rows.

Instructions:

- Find the rows most relevant to the user's question.
- Prioritize exact concept names, synonyms,
  acronyms, and phrase matches.
- Ignore unrelated rows.
- Do not say "no matching data was found"
  when relevant information exists in the
  supplied rows.
- Give a clear natural-language answer.
- Include the relevant definition when available.
- Be concise but informative.
""".strip(),
        },
        {
            "role": "user",
            "content": (
                f"QUESTION:\n{question}"
                f"\n\nQUERY RESULT:\n"
                f"{json.dumps(
                    compact_result,
                    indent=2,
                    default=str
                )}"
            ),
        },
    ]

    response = (
        client.chat.completions.create(
            model=ANSWER_MODEL,
            messages=messages,
        )
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


MAX_GENERATED_QUERY_ATTEMPTS = 3


def build_retry_context(question, attempts):
    return """
Previous generated query did not return usable graph rows.
Generate a broader Apache AGE Cypher query for the same question.
Use keyword search over safe scalar text properties when exact labels/relationships are uncertain.
Prefer OR conditions across related terms and synonyms.
Do not use OPTIONAL MATCH or toString().
Prior attempts:
{attempts_json}
""".strip().format(
        attempts_json=json.dumps(attempts, indent=2, default=str)
    )


def run_question_pipeline(question):
    attempts = []

    for attempt_number in range(1, MAX_GENERATED_QUERY_ATTEMPTS + 1):
        retry_context = None
        if attempt_number > 1:
            retry_context = build_retry_context(question, attempts)

        try:
            generated_spec = generate_age_query(question, retry_context=retry_context)
            generated_rows = execute_age_query(generated_spec)
            attempts.append({
                "stage": "generated",
                "attempt": attempt_number,
                "status": "success",
                "cypher": generated_spec["cypher"],
                "row_count": len(generated_rows),
            })
            if generated_rows:
                return generated_spec, generated_rows, attempts
        except Exception as exc:
            attempts.append({
                "stage": "generated",
                "attempt": attempt_number,
                "status": "failed",
                "error": str(exc),
            })

    fallback_spec = build_fallback_query(question)
    fallback_rows = execute_age_query(fallback_spec)
    attempts.append({
        "stage": "fallback",
        "attempt": 1,
        "status": "success",
        "cypher": fallback_spec["cypher"],
        "search_terms": fallback_spec.get("search_terms", []),
        "row_count": len(fallback_rows),
    })
    return fallback_spec, fallback_rows, attempts


def ask_age_graph(question, show_cypher=True, show_raw_result=False):
    query_spec, query_result, attempts = run_question_pipeline(question)
    answer = generate_answer(question, query_result, cypher=query_spec.get("cypher"))

    if show_cypher:
        print("Final Cypher:\n")
        print(query_spec["cypher"])
    if show_raw_result:
        print("\nPipeline Attempts:\n")
        print(json.dumps(attempts, indent=2, default=str))
        print("\nRaw AGE Result:\n")
        print(json.dumps(query_result, indent=2, default=str))
    return answer


In [13]:
print(ask_age_graph("Show the source evidence for UHC step therapy requirements"))

Search-term LLM failed; using deterministic terms only: name 'GRAPH_SCHEMA' is not defined
Final Cypher:

MATCH (n)
WHERE n.search_text CONTAINS 'show the source'
        OR n.search_text CONTAINS 'the source evidence'
        OR n.search_text CONTAINS 'source evidence for'
        OR n.search_text CONTAINS 'evidence for uhc'
        OR n.search_text CONTAINS 'for uhc step'
        OR n.search_text CONTAINS 'uhc step therapy'
        OR n.search_text CONTAINS 'step therapy requirements'
        OR n.search_text CONTAINS 'show the'
        OR n.search_text CONTAINS 'the source'
        OR n.search_text CONTAINS 'source evidence'
        OR n.search_text CONTAINS 'evidence for'
        OR n.search_text CONTAINS 'for uhc'
RETURN
    labels(n) AS labels,
    n.node_id AS node_id,
    n.title AS title,
    n.definition AS definition,
    n.description AS description,
    n.concept_id AS concept_id,
    n.entity_id AS entity_id,
    n.artifact_id AS artifact_id,
    n.artifact_type AS artifa

In [9]:
# Example questions after real_tranformation.ipynb has loaded the graph.
answer = ask_age_graph("List the available dataset entities and their domains", show_raw_result=True)
print(answer)


Search-term LLM failed; using deterministic terms only: name 'GRAPH_SCHEMA' is not defined
Final Cypher:

MATCH (n)
WHERE n.search_text CONTAINS 'list the available'
        OR n.search_text CONTAINS 'the available dataset'
        OR n.search_text CONTAINS 'available dataset entities'
        OR n.search_text CONTAINS 'dataset entities and'
        OR n.search_text CONTAINS 'entities and their'
        OR n.search_text CONTAINS 'and their domains'
        OR n.search_text CONTAINS 'list the'
        OR n.search_text CONTAINS 'the available'
        OR n.search_text CONTAINS 'available dataset'
        OR n.search_text CONTAINS 'dataset entities'
        OR n.search_text CONTAINS 'entities and'
        OR n.search_text CONTAINS 'and their'
RETURN
    labels(n) AS labels,
    n.node_id AS node_id,
    n.title AS title,
    n.definition AS definition,
    n.description AS description,
    n.concept_id AS concept_id,
    n.entity_id AS entity_id,
    n.artifact_id AS artifact_id,
    n.ar

In [11]:
answer = ask_age_graph("How can I setup MMM usecase?", show_raw_result=True)
print(answer)

Final Cypher:

MATCH (n) WHERE n.search_text CONTAINS 'marketing mix' OR n.search_text CONTAINS 'market mix' OR n.search_text CONTAINS 'media mix modeling' OR n.search_text CONTAINS 'market mix modeling' OR n.search_text CONTAINS 'marketing mix modeling' OR n.search_text CONTAINS 'marketing attribution' OR n.search_text CONTAINS 'incrementality' OR (n.search_text CONTAINS 'mmm' AND (n.search_text CONTAINS 'model' OR n.search_text CONTAINS 'marketing' OR n.search_text CONTAINS 'market' OR n.search_text CONTAINS 'media')) RETURN n.node_id AS node_id, n.search_text AS search_text, n.name AS name, n.title AS title, n.purpose AS purpose, n.definition AS definition, n.description AS description, n.routing AS routing, n.use_case_id AS use_case_id, n.step_number AS step_number, n.step_id AS step_id, n.slot_id AS slot_id, n.concept_id AS concept_id, n.dataset_id AS dataset_id LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "success",
    "cypher": "

In [12]:
answer = ask_age_graph("How can I calculate ROI?", show_raw_result=True)
print(answer)

Final Cypher:

MATCH (n) WHERE n.search_text CONTAINS 'return on investment' OR n.search_text CONTAINS 'investment return' OR n.search_text CONTAINS 'return on invested' OR n.search_text CONTAINS 'profitability' OR n.search_text CONTAINS 'cost benefit' OR n.search_text CONTAINS 'roi' RETURN n.node_id AS node_id, n.search_text AS search_text, n.name AS name, n.title AS title, n.purpose AS purpose, n.definition AS definition, n.description AS description, n.analysis AS analysis, n.business_rules AS business_rules LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "failed",
    "error": "syntax error at or near \":\"\nLINE 4:         MATCH (n) WHERE (n:Playbook OR n:Concept OR n:Datase...\n                                  ^"
  },
  {
    "stage": "generated",
    "attempt": 2,
    "status": "success",
    "cypher": "MATCH (n) WHERE n.search_text CONTAINS 'return on investment' OR n.search_text CONTAINS 'investment return' OR n.search_text CONTAI

In [13]:
answer = ask_age_graph("How Market share is calculated?", show_raw_result=True)
print(answer)

Final Cypher:

MATCH (n) WHERE n.search_text CONTAINS 'market share' OR n.search_text CONTAINS 'share of market' OR n.search_text CONTAINS 'market-size' OR n.search_text CONTAINS 'share calculation' OR n.search_text CONTAINS 'market share calculation' RETURN n.name AS name, n.title AS title, n.purpose AS purpose, n.definition AS definition, n.analysis AS analysis, n.business_rules AS business_rules, n.search_text AS search_text LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "failed",
    "error": "syntax error at or near \":\"\nLINE 4:         MATCH (n) WHERE (n:Concept OR n:Playbook) AND (n.sea...\n                                  ^"
  },
  {
    "stage": "generated",
    "attempt": 2,
    "status": "success",
    "cypher": "MATCH (n) WHERE n.search_text CONTAINS 'market share' OR n.search_text CONTAINS 'share of market' OR n.search_text CONTAINS 'market-size' OR n.search_text CONTAINS 'share calculation' OR n.search_text CONTAINS 'marke

In [14]:
answer = ask_age_graph("What is temperature in Hyderabad?", show_raw_result=True)
print(answer)

Final Cypher:

MATCH (n) WHERE n.search_text CONTAINS 'temperature' OR n.search_text CONTAINS 'weather' OR n.search_text CONTAINS 'climate' OR n.search_text CONTAINS 'hyderabad' OR n.search_text CONTAINS 'india' OR n.search_text CONTAINS 'city' OR n.search_text CONTAINS 'location' RETURN n.node_id AS node_id, n.artifact_id AS artifact_id, n.search_text AS search_text, n.source_file AS source_file LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "success",
    "cypher": "MATCH (d:Dataset) WHERE d.search_text CONTAINS 'temperature' AND d.search_text CONTAINS 'hyderabad' RETURN d.entity_id AS entity_id, d.domain AS domain, d.purpose AS purpose, d.gold_table AS gold_table LIMIT 10",
    "row_count": 0
  },
  {
    "stage": "generated",
    "attempt": 2,
    "status": "success",
    "cypher": "MATCH (n) WHERE n.search_text CONTAINS 'temperature' OR n.search_text CONTAINS 'weather' OR n.search_text CONTAINS 'climate' OR n.search_text CONTAINS 'hyd

In [15]:
answer = ask_age_graph("Define HCP 360 degree view analysis", show_raw_result=True)
print(answer)

Final Cypher:

MATCH (n) WHERE n.search_text CONTAINS 'healthcare provider' OR n.search_text CONTAINS 'healthcare professional' OR n.search_text CONTAINS 'hcp 360' OR n.search_text CONTAINS 'provider 360' OR n.search_text CONTAINS '360 degree view' OR n.search_text CONTAINS '360-degree view' OR n.search_text CONTAINS '360 view' RETURN n.name AS name, n.title AS title, n.definition AS definition, n.analysis AS analysis, n.purpose AS purpose, n.business_rules AS business_rules, n.narrative_rules AS narrative_rules, n.disambiguation_triggers AS disambiguation_triggers, n.what_it_is_not AS what_it_is_not LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "success",
    "cypher": "MATCH (c:Concept) WHERE c.search_text CONTAINS 'hcp 360' OR c.search_text CONTAINS 'healthcare provider 360' OR c.search_text CONTAINS '360 degree view' RETURN c.name AS concept_name, c.definition AS definition, c.disambiguation_triggers AS disambiguation_triggers, c.what

In [16]:
result = ask_age_graph("Can you tell me about LOT (Line of Therapy)?",show_raw_result=True)
print(result)

Final Cypher:

MATCH (c:Concept) WHERE c.search_text CONTAINS 'line of therapy' RETURN c.name AS name, c.definition AS definition, c.synonyms AS synonyms, c.what_it_is_not AS what_it_is_not LIMIT 10

Pipeline Attempts:

[
  {
    "stage": "generated",
    "attempt": 1,
    "status": "success",
    "cypher": "MATCH (c:Concept) WHERE c.search_text CONTAINS 'line of therapy' RETURN c.name AS name, c.definition AS definition, c.synonyms AS synonyms, c.what_it_is_not AS what_it_is_not LIMIT 10",
    "row_count": 3
  }
]

Raw AGE Result:

[
  {
    "name": "Line of Therapy (Sequential Regimen Logic)",
    "definition": "Line of therapy describes where a treatment sits in the sequence of\nregimens a patient has received \u2014 first-line, second-line, and so on. It\nis the most complex Patient IQ construct and is meaningful only for\nindications where sequential lines are clinically defined (notably some\noncology settings).\n\nWhat it captures: by ordering a patient's regimens over time and 

In [17]:
test_questions = [
    "What is HCP segmentation?",
    "What is Market Mix Modeling in the Commercial Pharma?",
    "How is RoI calculated in the Market Mix Modeling?",
    "What is specialty pharmacy?",
    "Give me information on the 340B sales program",
    "Can you tell me about LoT (Line of Therapy)?",
    "What is secondary market research?",
    "What is market access?",
    "What is the difference between qualitative and quantitative research?",
    "What is Gross-to-Net (GTN) in commercial pharma, and which deductions are usually included?",
    "How do formulary tiers, prior authorization, and step therapy affect market access?",
    "What is the difference between TRx, NRx, and NBRx in pharma analytics?",
    "What are patient support or hub services in specialty pharma, and how do they help with access and adherence"
]

In [18]:
def ask_age_graph_record(question):
    record = {
        "question": question,
        "answer": "",
        "cypher query": "",
        "structured response": "",
    }

    query_spec = None
    attempts = []
    try:
        query_spec, query_result, attempts = run_question_pipeline(question)
        answer = generate_answer(question, query_result,cypher=query_spec["cypher"])

        record["answer"] = answer
        record["cypher query"] = query_spec["cypher"]
        record["structured response"] = json.dumps(
            {
                "columns": query_spec["columns"],
                "rows": compact_query_result(query_result),
                "attempts": attempts,
            },
            default=str,
            ensure_ascii=False,
        )
    except Exception as exc:
        record["answer"] = f"ERROR: {exc}"
        if query_spec:
            record["cypher query"] = query_spec.get("cypher", "")
        record["structured response"] = json.dumps(
            {
                "error": str(exc),
                "query_spec": query_spec,
                "attempts": attempts,
            },
            default=str,
            ensure_ascii=False,
        )

    return record


def export_question_answers_to_csv(questions, csv_path="real_graph_qa_results.csv"):
    fieldnames = [
        "question",
        "answer",
        "cypher query",
        "structured response",
    ]

    records = []
    for index, question in enumerate(questions, start=1):
        print(f"Running {index}/{len(questions)}: {question}")
        records.append(ask_age_graph_record(question))

    with open(csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)

    print("Saved CSV:", csv_path)
    return records


# Run this cell after `test_questions` is defined.
qa_records = export_question_answers_to_csv(
    test_questions,
    csv_path="real_graph_qa_results.csv",
)
qa_records[:2]


Running 1/13: What is HCP segmentation?
Running 2/13: What is Market Mix Modeling in the Commercial Pharma?
Running 3/13: How is RoI calculated in the Market Mix Modeling?
Running 4/13: What is specialty pharmacy?
Running 5/13: Give me information on the 340B sales program
Running 6/13: Can you tell me about LoT (Line of Therapy)?
Running 7/13: What is secondary market research?
Running 8/13: What is market access?
Running 9/13: What is the difference between qualitative and quantitative research?
Running 10/13: What is Gross-to-Net (GTN) in commercial pharma, and which deductions are usually included?
Running 11/13: How do formulary tiers, prior authorization, and step therapy affect market access?
Running 12/13: What is the difference between TRx, NRx, and NBRx in pharma analytics?
Running 13/13: What are patient support or hub services in specialty pharma, and how do they help with access and adherence
Saved CSV: real_graph_qa_results.csv


[{'question': 'What is HCP segmentation?',
  'answer': 'HCP segmentation is the process of grouping healthcare professionals (HCPs) into strategically meaningful segments based on characteristics such as prescribing behavior, brand share, market volume, or targeting metrics.\n\nA common approach is **HCP Share × Volume Segmentation**, which groups HCPs by:\n\n- **High share:** Loyal, high-value brand prescribers  \n- **Low share, high market volume:** High-volume prescribers who favor competitors and represent key growth targets  \n- **Low share, low market volume:** Lower-value prescribers who may require development or conversion efforts  \n- **No prescriptions:** HCPs with no brand or market prescribing in the base period  \n\nAnother approach is **HCP decile segmentation**, which ranks prescribers by a defined metric and divides them into 10 groups, with D10 representing the highest-ranked group.',
  'cypher query': "MATCH (n:Concept) WHERE n.search_text CONTAINS 'hcp segmentation'